# Notebook 03 — Retry Policies

This notebook covers Activity retry policies — how Temporal automatically retries failed Activities.

## What You Will Learn
- Default retry behavior
- How to configure custom retry policies
- Exponential backoff
- Non-retryable errors

## Default Retry Behavior

Without an explicit retry policy, Temporal retries Activities with:

| Setting | Default |
|---------|---------|
| Initial Interval | 1 second |
| Backoff Coefficient | 2.0 |
| Maximum Interval | 100 × initial interval |
| Maximum Attempts | **Unlimited** |

⚠️ Activities retry **forever** by default.

In [ ]:
from datetime import timedelta
from temporalio.common import RetryPolicy

# Custom retry policy
retry_policy = RetryPolicy(
    initial_interval=timedelta(seconds=15),
    backoff_coefficient=2.0,
    maximum_interval=timedelta(seconds=160),
    maximum_attempts=100,
)

print(f"Initial interval: {retry_policy.initial_interval}")
print(f"Backoff coefficient: {retry_policy.backoff_coefficient}")
print(f"Maximum interval: {retry_policy.maximum_interval}")
print(f"Maximum attempts: {retry_policy.maximum_attempts}")

## Retry Timing Visualization

With `initial_interval=15s` and `backoff_coefficient=2.0`:

```
Attempt 1: immediate
Attempt 2: wait 15s
Attempt 3: wait 30s
Attempt 4: wait 60s
Attempt 5: wait 120s
Attempt 6: wait 160s  (capped)
Attempt 7: wait 160s  (capped)
```

In [ ]:
# Simulate exponential backoff timing
initial = 15
coefficient = 2.0
max_interval = 160
max_attempts = 10

interval = initial
total_wait = 0

print(f"{'Attempt':<10} {'Wait (s)':<12} {'Total (s)':<12}")
print("-" * 34)
print(f"{'1':<10} {'0':<12} {'0':<12}")

for attempt in range(2, max_attempts + 1):
    total_wait += interval
    print(f"{attempt:<10} {interval:<12} {total_wait:<12}")
    interval = min(interval * coefficient, max_interval)

## Applying a Retry Policy in a Workflow

In [ ]:
from datetime import timedelta
from temporalio import workflow
from temporalio.common import RetryPolicy


# Example Workflow with retry policy
@workflow.defn
class GreetWithRetry:
    @workflow.run
    async def run(self, name: str) -> str:
        retry_policy = RetryPolicy(
            initial_interval=timedelta(seconds=15),
            backoff_coefficient=2.0,
            maximum_interval=timedelta(seconds=160),
            maximum_attempts=100,
        )

        # Activity execution with retry policy
        # greeting = await workflow.execute_activity(
        #     greet_in_spanish,
        #     name,
        #     start_to_close_timeout=timedelta(seconds=5),
        #     retry_policy=retry_policy,
        # )
        # return greeting
        return f"Would greet {name} with retry policy"


print("Workflow with retry policy defined")

## Non-Retryable Errors

Some errors should not be retried (e.g., invalid input):

In [ ]:
from temporalio.exceptions import ApplicationError


# This error will NOT be retried
try:
    raise ApplicationError(
        "Invalid order amount",
        non_retryable=True,
    )
except ApplicationError as e:
    print(f"Error: {e}")
    print(f"Non-retryable: {e.non_retryable}")

## Monitoring Retries in the Web UI

When an Activity is retried, the event history shows:
```
ActivityTaskScheduled
ActivityTaskStarted
ActivityTaskFailed       ← Attempt 1 failed
ActivityTaskScheduled    ← Retry scheduled
ActivityTaskStarted
ActivityTaskCompleted    ← Attempt 2 succeeded
```

Open `http://localhost:8233` and click on a Workflow to see its event history.

## Summary

- Temporal retries failed Activities **automatically**
- Default: unlimited retries with exponential backoff
- Use `RetryPolicy` to customize retry behavior
- Use `ApplicationError(non_retryable=True)` for errors that should not be retried
- Monitor retries in the Temporal Web UI event history

**Next:** [04_durable_execution.ipynb](04_durable_execution.ipynb)